In [ ]:
# An attempt to identify the correct file for a given video using MSE and 4 control points
import numpy as np
import pandas as pd
from scipy.optimize import least_squares

IMG_WIDTH, IMG_HEIGHT = 1920, 1080
VIDEO_FPS = 60
GCP_FRAME = 741   # a frame showing all 4 control points marked

TELEMETRY_FILES = [
    "/kaggle/input/datasets/natair/djiflightrecord-2026-04-02-16-20-13-csv/DJIFlightRecord_2026-04-02_16-20-13.csv",
    "/kaggle/input/datasets/natair/djiflightrecord-2026-04-02-17-00-04-1/DJIFlightRecord_2026-04-02_17-00-04.csv",
    "/kaggle/input/datasets/natair/djiflightrecord-2026-04-02-17-50-03-csv/DJIFlightRecord_2026-04-02_17-50-03.csv",
]

GROUND_CONTROL_POINTS = [
    {"frame": GCP_FRAME, "px": 500,  "py": 220, "lat": 52.75236395813839, "lon": 13.680075477949087},
    {"frame": GCP_FRAME, "px": 1750, "py": 700, "lat": 52.75142022264314, "lon": 13.68265678136193},
    {"frame": GCP_FRAME, "px": 1050, "py": 630, "lat": 52.75073848839966, "lon": 13.6822158755006},
    {"frame": GCP_FRAME, "px": 0,    "py": 250, "lat": 52.75163129122488, "lon": 13.6794141191636},
]

def latlng_to_local_m(lat, lng, ref_lat, ref_lng):
    m_per_deg_lat = 111320.0
    m_per_deg_lng = 111320.0 * np.cos(np.radians(ref_lat))
    east = (lng - ref_lng) * m_per_deg_lng
    north = (lat - ref_lat) * m_per_deg_lat
    return east, north

def evaluate_telemetry_file(csv_path, gcps):
    try:
        telemetry = pd.read_csv(csv_path)
        telemetry = telemetry[["time_s", "lat", "lng", "altitude_m", "yaw_deg"]].dropna()
    except Exception as e:
        return None, None, f"File read error: {e}"

    if len(telemetry) == 0:
        return None, None, "Empty telemetry after NaN cleanup"

    ref_lat, ref_lng = telemetry.iloc[0]["lat"], telemetry.iloc[0]["lng"]

    def get_drone_state(time_s):
        idx = (telemetry["time_s"] - time_s).abs().idxmin()
        row = telemetry.loc[idx]
        return row["lat"], row["lng"], row["altitude_m"], row["yaw_deg"]

    def residuals(params, gcps):
        focal_px = params[0]
        errs = []
        for gcp in gcps:
            lat, lng, alt, yaw = get_drone_state(gcp["frame"] / VIDEO_FPS)
            dx_px = gcp["px"] - IMG_WIDTH / 2
            dy_px = gcp["py"] - IMG_HEIGHT / 2

            yaw_rad = np.radians(yaw)
            east_offset  = (dx_px * np.cos(yaw_rad) + dy_px * np.sin(yaw_rad)) * alt / focal_px
            north_offset = (-dx_px * np.sin(yaw_rad) + dy_px * np.cos(yaw_rad)) * alt / focal_px

            drone_east, drone_north = latlng_to_local_m(lat, lng, ref_lat, ref_lng)
            pred_east  = drone_east + east_offset
            pred_north = drone_north - north_offset

            true_east, true_north = latlng_to_local_m(gcp["lat"], gcp["lon"], ref_lat, ref_lng)
            errs.extend([pred_east - true_east, pred_north - true_north])
        return errs

    try:
        result = least_squares(residuals, x0=[1200.0], args=(gcps,), max_nfev=5000)
    except Exception as e:
        return None, None, f"The optimization failed: {e}"

    focal_px = result.x[0]
    final_errors = np.array(residuals([focal_px], gcps)).reshape(-1, 2)
    rmse_m = np.sqrt((final_errors**2).sum(axis=1)).mean()

    lat0, lng0, alt0, yaw0 = get_drone_state(GCP_FRAME / VIDEO_FPS)

    diag = f"altitude={alt0:.1f}m, yaw={yaw0:.1f}°, flight_time={GCP_FRAME/VIDEO_FPS:.1f}s"

    return focal_px, rmse_m, diag

print(f" Checking {len(TELEMETRY_FILES)} telemetry files in frame {GCP_FRAME}")

results = []

for csv_path in TELEMETRY_FILES:
    fname = csv_path.split("/")[-1]
    focal_px, rmse_m, diag = evaluate_telemetry_file(csv_path, GROUND_CONTROL_POINTS)

    print(f"{fname}")
    if focal_px is None:
        print(f" {diag}\n")
        continue

    print(f"  The drone's status in the frame {GCP_FRAME}: {diag}")
    print(f"  Calibrated Focus: {focal_px:.1f} px")
    print(f"  RMSE errors:      {rmse_m:.2f} meters")
    results.append((fname, focal_px, rmse_m))

if results:
    results_sorted = sorted(results, key=lambda x: x[2])
    for fname, focal_px, rmse_m in results_sorted:
        print(f" RMSE={rmse_m:6.2f}м | focal={focal_px:7.1f}px | {fname}")

    best = results_sorted[0]
    print(f"\n The most likely correct file: {best[0]}")
    print(f" (smallest RMSE error = {best[2]:.2f}m)")
else:
    print("None of the files yielded any results")


In [ ]:
# Search for a Telemetry and Time Shift File
import numpy as np
import pandas as pd
from scipy.optimize import least_squares

IMG_WIDTH, IMG_HEIGHT = 1920, 1080
VIDEO_FPS = 60
GCP_FRAME = 741
VIDEO_DURATION_S = 420

TELEMETRY_FILES = [
    "/kaggle/input/datasets/natair/djiflightrecord-2026-04-02-16-20-13-csv/DJIFlightRecord_2026-04-02_16-20-13.csv",
    "/kaggle/input/datasets/natair/djiflightrecord-2026-04-02-17-00-04-1/DJIFlightRecord_2026-04-02_17-00-04.csv",
    "/kaggle/input/datasets/natair/djiflightrecord-2026-04-02-17-50-03-csv/DJIFlightRecord_2026-04-02_17-50-03.csv",
]

GROUND_CONTROL_POINTS = [
    {"frame": GCP_FRAME, "px": 500,  "py": 220, "lat": 52.75236395813839, "lon": 13.680075477949087},
    {"frame": GCP_FRAME, "px": 1750, "py": 700, "lat": 52.75142022264314, "lon": 13.68265678136193},
    {"frame": GCP_FRAME, "px": 1050, "py": 630, "lat": 52.75073848839966, "lon": 13.6822158755006},
    {"frame": GCP_FRAME, "px": 0,    "py": 250, "lat": 52.75163129122488, "lon": 13.6794141191636},
]

def latlng_to_local_m(lat, lng, ref_lat, ref_lng):
    m_per_deg_lat = 111320.0
    m_per_deg_lng = 111320.0 * np.cos(np.radians(ref_lat))
    east = (lng - ref_lng) * m_per_deg_lng
    north = (lat - ref_lat) * m_per_deg_lat
    return east, north

def evaluate_at_offset(telemetry, ref_lat, ref_lng, gcps, offset_s):
    """Calculates the calibration RMSE for a specific time shift from video to telemetry"""

    def get_drone_state(time_s):
        idx = (telemetry["time_s"] - time_s).abs().idxmin()
        row = telemetry.loc[idx]
        return row["lat"], row["lng"], row["altitude_m"], row["yaw_deg"]

    def residuals(params):
        focal_px = params[0]
        errs = []
        for gcp in gcps:
            video_time = gcp["frame"] / VIDEO_FPS
            telemetry_time = video_time + offset_s
            lat, lng, alt, yaw = get_drone_state(telemetry_time)

            dx_px = gcp["px"] - IMG_WIDTH / 2
            dy_px = gcp["py"] - IMG_HEIGHT / 2
            yaw_rad = np.radians(yaw)
            east_offset  = (dx_px * np.cos(yaw_rad) + dy_px * np.sin(yaw_rad)) * alt / focal_px
            north_offset = (-dx_px * np.sin(yaw_rad) + dy_px * np.cos(yaw_rad)) * alt / focal_px

            drone_east, drone_north = latlng_to_local_m(lat, lng, ref_lat, ref_lng)
            pred_east  = drone_east + east_offset
            pred_north = drone_north - north_offset

            true_east, true_north = latlng_to_local_m(gcp["lat"], gcp["lon"], ref_lat, ref_lng)
            errs.extend([pred_east - true_east, pred_north - true_north])
        return errs

    try:
        result = least_squares(residuals, x0=[1200.0], max_nfev=2000)
        focal_px = result.x[0]
        final_errors = np.array(residuals([focal_px])).reshape(-1, 2)
        rmse_m = np.sqrt((final_errors**2).sum(axis=1)).mean()
        return focal_px, rmse_m
    except Exception:
        return None, np.inf


# Complete scan: file × time shift
all_results = []

for csv_path in TELEMETRY_FILES:
    fname = csv_path.split("/")[-1]
    telemetry = pd.read_csv(csv_path)[["time_s", "lat", "lng", "altitude_m", "yaw_deg"]].dropna()
    ref_lat, ref_lng = telemetry.iloc[0]["lat"], telemetry.iloc[0]["lng"]
    flight_duration = telemetry["time_s"].max()

    max_offset = max(0, flight_duration - VIDEO_DURATION_S)
    coarse_offsets = np.arange(-30, max_offset + 30, 1.0)

    best_rmse = np.inf
    best_offset = None
    best_focal = None

    for offset in coarse_offsets:
        focal_px, rmse_m = evaluate_at_offset(telemetry, ref_lat, ref_lng, GROUND_CONTROL_POINTS, offset)
        if focal_px is not None and 300 < focal_px < 5000 and rmse_m < best_rmse:
            best_rmse = rmse_m
            best_offset = offset
            best_focal = focal_px

    if best_offset is not None:
        fine_offsets = np.arange(best_offset - 1, best_offset + 1, 0.1)
        for offset in fine_offsets:
            focal_px, rmse_m = evaluate_at_offset(telemetry, ref_lat, ref_lng, GROUND_CONTROL_POINTS, offset)
            if focal_px is not None and 300 < focal_px < 5000 and rmse_m < best_rmse:
                best_rmse = rmse_m
                best_offset = offset
                best_focal = focal_px

    print(f"{fname}")
    if best_offset is not None:
        print(f" Best offset: {best_offset:+.1f}s  |  RMSE: {best_rmse:.2f}m  |  focal: {best_focal:.0f}px")
    else:
        print(f" No reasonable combination was found (the entire range results in a focal length outside the 300-5,000 px range)")
    print()

    all_results.append((fname, best_offset, best_focal, best_rmse))

valid_results = [r for r in all_results if r[1] is not None]
valid_results.sort(key=lambda x: x[3])

for fname, offset, focal, rmse in valid_results:
    print(f" RMSE={rmse:6.2f}m | offset={offset:+7.1f}s | focal={focal:7.0f}px | {fname}")

if valid_results:
    best = valid_results[0]
    print(f"\n Best: {best[0]}")
    print(f" offset = {best[1]:+.1f} sec")
    print(f" RMSE = {best[3]:.2f}m")

In [ ]:
# Retrieving Telemetry via Independent Estimation of the Drone's Pose
import numpy as np
import pandas as pd
from scipy.optimize import least_squares

IMG_WIDTH, IMG_HEIGHT = 1920, 1080
GCP_FRAME = 741

TELEMETRY_FILES = [
    "/kaggle/input/datasets/natair/djiflightrecord-2026-04-02-16-20-13-csv/DJIFlightRecord_2026-04-02_16-20-13.csv",
    "/kaggle/input/datasets/natair/djiflightrecord-2026-04-02-17-00-04-1/DJIFlightRecord_2026-04-02_17-00-04.csv",
    "/kaggle/input/datasets/natair/djiflightrecord-2026-04-02-17-50-03-csv/DJIFlightRecord_2026-04-02_17-50-03.csv",
]

GROUND_CONTROL_POINTS = [
    {"px": 500,  "py": 220, "lat": 52.75236395813839, "lon": 13.680075477949087},
    {"px": 1750, "py": 700, "lat": 52.75142022264314, "lon": 13.68265678136193},
    {"px": 1050, "py": 630, "lat": 52.75073848839966, "lon": 13.6822158755006},
    {"px": 0,    "py": 250, "lat": 52.75163129122488, "lon": 13.6794141191636},
]

def latlng_to_local_m(lat, lng, ref_lat, ref_lng):
    m_per_deg_lat = 111320.0
    m_per_deg_lng = 111320.0 * np.cos(np.radians(ref_lat))
    east = (lng - ref_lng) * m_per_deg_lng
    north = (lat - ref_lat) * m_per_deg_lat
    return east, north

def local_m_to_latlng(east, north, ref_lat, ref_lng):
    m_per_deg_lat = 111320.0
    m_per_deg_lng = 111320.0 * np.cos(np.radians(ref_lat))
    lat = ref_lat + north / m_per_deg_lat
    lng = ref_lng + east / m_per_deg_lng
    return lat, lng

REF_LAT = np.mean([g["lat"] for g in GROUND_CONTROL_POINTS])
REF_LNG = np.mean([g["lon"] for g in GROUND_CONTROL_POINTS])

def pose_residuals(params, gcps):
    drone_east, drone_north, altitude, yaw_deg, focal_px = params
    yaw_rad = np.radians(yaw_deg)
    errs = []
    for gcp in gcps:
        dx_px = gcp["px"] - IMG_WIDTH / 2
        dy_px = gcp["py"] - IMG_HEIGHT / 2
        east_offset  = (dx_px * np.cos(yaw_rad) + dy_px * np.sin(yaw_rad)) * altitude / focal_px
        north_offset = (-dx_px * np.sin(yaw_rad) + dy_px * np.cos(yaw_rad)) * altitude / focal_px

        pred_east  = drone_east + east_offset
        pred_north = drone_north - north_offset

        true_east, true_north = latlng_to_local_m(gcp["lat"], gcp["lon"], REF_LAT, REF_LNG)
        errs.extend([pred_east - true_east, pred_north - true_north])
    return errs

x0 = [0, 0, 35.0, 0.0, 1200.0]

result = least_squares(
    pose_residuals, x0=x0, args=(GROUND_CONTROL_POINTS,),
    max_nfev=20000,
    bounds=([-500, -500, 5, -180, 200], [500, 500, 150, 180, 5000]),
)

drone_east, drone_north, est_altitude, est_yaw, est_focal = result.x
est_lat, est_lng = local_m_to_latlng(drone_east, drone_north, REF_LAT, REF_LNG)

final_errors = np.array(pose_residuals(result.x, GROUND_CONTROL_POINTS)).reshape(-1, 2)
rmse_m = np.sqrt((final_errors**2).sum(axis=1)).mean()

print(f"  Drone position estimate for frame {GCP_FRAME} (from GCP only, without telemetry)")
print(f"  Position:    lat={est_lat:.7f}, lon={est_lng:.7f}")
print(f"  Altitude:     {est_altitude:.1f} m")
print(f"  Yaw: {est_yaw:.1f}°")
print(f"  Focal length:      {est_focal:.0f} px")
print(f"  RMSE of GCP fit: {rmse_m:.2f} m")

print(f"  Search for the closest match across all telemetry data")
all_matches = []

for csv_path in TELEMETRY_FILES:
    fname = csv_path.split("/")[-1]
    telemetry = pd.read_csv(csv_path)[["time_s", "lat", "lng", "altitude_m", "yaw_deg"]].dropna()

    # Calculate the "distance" of each telemetry line to the estimated position:
    # position error (meters) + penalty for altitude difference + penalty for heading difference
    east_t, north_t = latlng_to_local_m(
        telemetry["lat"].values, telemetry["lng"].values, REF_LAT, REF_LNG
    )
    pos_dist = np.hypot(east_t - drone_east, north_t - drone_north)
    alt_diff = np.abs(telemetry["altitude_m"].values - est_altitude)

    yaw_diff = np.abs(((telemetry["yaw_deg"].values - est_yaw + 180) % 360) - 180)

    combined = pos_dist + alt_diff * 2 + yaw_diff * 0.3

    best_idx = np.argmin(combined)
    best_row = telemetry.iloc[best_idx]

    print(f"{fname}")
    print(f"  Best match: t={best_row[‘time_s’]:.1f} s")
    print(f"    Position error: {pos_dist[best_idx]:.1f} m")
    print(f"    Altitude difference: {alt_diff[best_idx]:.1f} m  (telemetry={best_row[‘altitude_m’]:.1f} m, estimate={est_altitude:.1f} m)")
    print(f"    Yaw difference: {yaw_diff[best_idx]:.1f}°  (telemetry={best_row[‘yaw_deg’]:.1f}°, estimate={est_yaw:.1f}°)")
    print()

    all_matches.append({
        "file": fname, "time_s": best_row["time_s"],
        "pos_dist": pos_dist[best_idx], "alt_diff": alt_diff[best_idx],
        "yaw_diff": yaw_diff[best_idx], "combined": combined[best_idx],
    })

all_matches.sort(key=lambda x: x["combined"])
for m in all_matches:
    print(f"  {m['file']}: t={m['time_s']:.1f}s, "
          f"position={m['pos_dist']:.1f}m, height_difference={m['alt_diff']:.1f}m, course_variance={m['yaw_diff']:.1f}°")

best = all_matches[0]
video_time_at_gcp = GCP_FRAME / 60
offset = best["time_s"] - video_time_at_gcp

print(f"\n Best: {best['file']}")
print(f"    Telemetry t={best["time_s"]:.1f}s corresponds to video frame {GCP_FRAME} (video time {video_time_at_gcp:.1f}s)")
print(f"    offset = {offset:+.1f} sec")

In [ ]:
# Visual Check for Compliance with GCP 4
import matplotlib.pyplot as plt
import numpy as np

GROUND_CONTROL_POINTS = [
    {"name": "distant right", "px": 500,  "py": 220, "lat": 52.75236395813839, "lon": 13.680075477949087},
    {"name": "close right","px": 1750, "py": 700, "lat": 52.75142022264314, "lon": 13.68265678136193},
    {"name": "close left", "px": 1050, "py": 630, "lat": 52.75073848839966, "lon": 13.6822158755006},
    {"name": "distant left", "px": 0,    "py": 250, "lat": 52.75163129122488, "lon": 13.6794141191636},
]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

ax = axes[0]
pxs = [g["px"] for g in GROUND_CONTROL_POINTS] + [GROUND_CONTROL_POINTS[0]["px"]]
pys = [g["py"] for g in GROUND_CONTROL_POINTS] + [GROUND_CONTROL_POINTS[0]["py"]]
ax.plot(pxs, pys, 'o-', color='blue')
for i, g in enumerate(GROUND_CONTROL_POINTS):
    ax.annotate(f"{i+1}. {g['name']}", (g["px"], g["py"]), fontsize=10,
                xytext=(10, 10), textcoords='offset points')
ax.set_xlim(-100, 2000)
ax.set_ylim(1150, -100)
ax.set_xlabel("px")
ax.set_ylabel("py")
ax.grid(alpha=0.3)

ax = axes[1]
lons = [g["lon"] for g in GROUND_CONTROL_POINTS] + [GROUND_CONTROL_POINTS[0]["lon"]]
lats = [g["lat"] for g in GROUND_CONTROL_POINTS] + [GROUND_CONTROL_POINTS[0]["lat"]]
ax.plot(lons, lats, 'o-', color='green')
for i, g in enumerate(GROUND_CONTROL_POINTS):
    ax.annotate(f"{i+1}. {g['name']}", (g["lon"], g["lat"]), fontsize=10,
                xytext=(10, 10), textcoords='offset points')
ax.set_xlabel("lon")
ax.set_ylabel("lat")
ax.grid(alpha=0.3)
ax.set_aspect('equal')

plt.tight_layout()
plt.savefig("/kaggle/working/gcp_sanity_check.jpg", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# Leave-one-out: Identifying the problematic GCP point
import numpy as np
from scipy.optimize import least_squares
from itertools import combinations

IMG_WIDTH, IMG_HEIGHT = 1920, 1080

GROUND_CONTROL_POINTS = [
    {"name": "distant right", "px": 500,  "py": 220, "lat": 52.75236395813839, "lon": 13.680075477949087},
    {"name": "close left", "px": 1050, "py": 630, "lat": 52.75073848839966, "lon": 13.6822158755006},
    {"name": "close right", "px": 1750, "py": 700, "lat": 52.75142022264314, "lon": 13.68265678136193},
    {"name": "distant left", "px": 0,    "py": 250, "lat": 52.75163129122488, "lon": 13.6794141191636},
]

def latlng_to_local_m(lat, lng, ref_lat, ref_lng):
    m_per_deg_lat = 111320.0
    m_per_deg_lng = 111320.0 * np.cos(np.radians(ref_lat))
    east = (lng - ref_lng) * m_per_deg_lng
    north = (lat - ref_lat) * m_per_deg_lat
    return east, north

def fit_pose(gcps):
    ref_lat = np.mean([g["lat"] for g in gcps])
    ref_lng = np.mean([g["lon"] for g in gcps])

    def residuals(params):
        drone_east, drone_north, altitude, yaw_deg, focal_px = params
        yaw_rad = np.radians(yaw_deg)
        errs = []
        for gcp in gcps:
            dx_px = gcp["px"] - IMG_WIDTH / 2
            dy_px = gcp["py"] - IMG_HEIGHT / 2
            east_offset  = (dx_px * np.cos(yaw_rad) + dy_px * np.sin(yaw_rad)) * altitude / focal_px
            north_offset = (-dx_px * np.sin(yaw_rad) + dy_px * np.cos(yaw_rad)) * altitude / focal_px
            pred_east  = drone_east + east_offset
            pred_north = drone_north - north_offset
            true_east, true_north = latlng_to_local_m(gcp["lat"], gcp["lon"], ref_lat, ref_lng)
            errs.extend([pred_east - true_east, pred_north - true_north])
        return errs

    x0 = [0, 0, 35.0, 0.0, 1200.0]
    result = least_squares(residuals, x0=x0, max_nfev=20000,
                            bounds=([-500, -500, 5, -180, 200], [500, 500, 150, 180, 5000]))
    final_errors = np.array(residuals(result.x)).reshape(-1, 2)
    rmse_m = np.sqrt((final_errors**2).sum(axis=1)).mean()
    return result.x, rmse_m

_, base_rmse = fit_pose(GROUND_CONTROL_POINTS)
print(f" RMSE = {base_rmse:.2f}m\n")
print(f"  Leave-one-out: Remove one point at a time")

results = []
for i, excluded in enumerate(GROUND_CONTROL_POINTS):
    subset = [g for j, g in enumerate(GROUND_CONTROL_POINTS) if j != i]
    params, rmse_m = fit_pose(subset)
    drone_east, drone_north, alt, yaw, focal = params
    print(f"Without '{excluded['name']}':")
    print(f" RMSE = {rmse_m:.2f}m  |  height = {alt:.1f}m  |  focus = {focal:.0f}px  |  yaw = {yaw:.1f}°")
    results.append((excluded["name"], rmse_m, alt, focal))
    print()

results.sort(key=lambda x: x[1])
for name, rmse_m, alt, focal in results:
    marker = " Problematic dot!" if rmse_m == results[0][1] else ""
    print(f"  Without '{name}': RMSE={rmse_m:6.2f}m, height={alt:5.1f}m, focus={focal:6.0f}px{marker}")

In [ ]:
# Full model accounting for camera tilt
import numpy as np
from scipy.optimize import least_squares

IMG_WIDTH, IMG_HEIGHT = 1920, 1080

GROUND_CONTROL_POINTS = [
    {"name": "distant right", "px": 500,  "py": 220, "lat": 52.75236395813839, "lon": 13.680075477949087},
    {"name": "close left", "px": 1050, "py": 630, "lat": 52.75073848839966, "lon": 13.6822158755006},
    {"name": "close right", "px": 1750, "py": 700, "lat": 52.75142022264314, "lon": 13.68265678136193},
    {"name": "distant left", "px": 0,    "py": 250, "lat": 52.75163129122488, "lon": 13.6794141191636},
]

def latlng_to_local_m(lat, lng, ref_lat, ref_lng):
    m_per_deg_lat = 111320.0
    m_per_deg_lng = 111320.0 * np.cos(np.radians(ref_lat))
    east = (lng - ref_lng) * m_per_deg_lng
    north = (lat - ref_lat) * m_per_deg_lat
    return east, north

ref_lat = np.mean([g["lat"] for g in GROUND_CONTROL_POINTS])
ref_lng = np.mean([g["lon"] for g in GROUND_CONTROL_POINTS])

def project_ray_to_ground(drone_east, drone_north, altitude, yaw_deg, pitch_down_deg, focal_px, px, py):
    """
    pitch_down: 90° = the camera is pointing straight down, 0° = pointing horizontally forward
    full model - the camera's beam intersects the ground plane at z=0
    """
    yaw = np.radians(yaw_deg)
    pitch = np.radians(pitch_down_deg)

    forward = np.array([np.sin(yaw) * np.cos(pitch), np.cos(yaw) * np.cos(pitch), -np.sin(pitch)])
    right   = np.array([np.cos(yaw), -np.sin(yaw), 0])
    down    = np.cross(forward, right)

    dx_px = px - IMG_WIDTH / 2
    dy_px = py - IMG_HEIGHT / 2

    direction = right * (dx_px / focal_px) + down * (dy_px / focal_px) + forward
    direction = direction / np.linalg.norm(direction)

    if direction[2] >= -1e-6:
        return None

    t = -altitude / direction[2]
    ground_east  = drone_east + t * direction[0]
    ground_north = drone_north + t * direction[1]
    return ground_east, ground_north

def residuals(params, gcps):
    drone_east, drone_north, altitude, yaw_deg, pitch_down_deg, focal_px = params
    errs = []
    for gcp in gcps:
        proj = project_ray_to_ground(drone_east, drone_north, altitude, yaw_deg, pitch_down_deg, focal_px,
                                      gcp["px"], gcp["py"])
        if proj is None:
            errs.extend([1000, 1000])
            continue
        pred_east, pred_north = proj
        true_east, true_north = latlng_to_local_m(gcp["lat"], gcp["lon"], ref_lat, ref_lng)
        errs.extend([pred_east - true_east, pred_north - true_north])
    return errs

best_result = None
best_rmse = np.inf

for pitch_guess in [90, 80, 70, 60, 50, 40, 30]:
    x0 = [0, 0, 30.0, 0.0, pitch_guess, 1200.0]
    try:
        result = least_squares(
            residuals, x0=x0, args=(GROUND_CONTROL_POINTS,),
            max_nfev=20000,
            bounds=([-500, -500, 5, -180, 5, 200], [500, 500, 150, 180, 90, 5000]),
        )
        final_errors = np.array(residuals(result.x, GROUND_CONTROL_POINTS)).reshape(-1, 2)
        rmse_m = np.sqrt((final_errors**2).sum(axis=1)).mean()
        print(f"  pitch_guess={pitch_guess}°: final pitch={result.x[4]:.1f}°, alt={result.x[2]:.1f}m, RMSE={rmse_m:.2f}m")
        if rmse_m < best_rmse:
            best_rmse = rmse_m
            best_result = result
    except Exception as e:
        print(f" pitch_guess={pitch_guess}°: error {e}")

drone_east, drone_north, altitude, yaw_deg, pitch_down_deg, focal_px = best_result.x
print(f"  Altitude:        {altitude:.1f} m")
print(f"  Yaw:     {yaw_deg:.1f}°")
print(f"  Camera pitch:  {pitch_down_deg:.1f}°  (90° = straight down, 0° = horizontal)")
print(f"  Focus:          {focal_px:.0f} px")
print(f"  RMSE:           {best_rmse:.2f} m")

In [ ]:
# Search for a telemetry file with a CORRECTED attitude (taking tilt into account)
import numpy as np
import pandas as pd

IMG_WIDTH, IMG_HEIGHT = 1920, 1080

TELEMETRY_FILES = [
    "/kaggle/input/datasets/natair/djiflightrecord-2026-04-02-16-20-13-csv/DJIFlightRecord_2026-04-02_16-20-13.csv",
    "/kaggle/input/datasets/natair/djiflightrecord-2026-04-02-17-00-04-1/DJIFlightRecord_2026-04-02_17-00-04.csv",
    "/kaggle/input/datasets/natair/djiflightrecord-2026-04-02-17-50-03-csv/DJIFlightRecord_2026-04-02_17-50-03.csv",
]

def latlng_to_local_m(lat, lng, ref_lat, ref_lng):
    m_per_deg_lat = 111320.0
    m_per_deg_lng = 111320.0 * np.cos(np.radians(ref_lat))
    east = (lng - ref_lng) * m_per_deg_lng
    north = (lat - ref_lat) * m_per_deg_lat
    return east, north

def local_m_to_latlng(east, north, ref_lat, ref_lng):
    m_per_deg_lat = 111320.0
    m_per_deg_lng = 111320.0 * np.cos(np.radians(ref_lat))
    lat = ref_lat + north / m_per_deg_lat
    lng = ref_lng + east / m_per_deg_lng
    return lat, lng

REF_LAT = 52.751596756617244
REF_LNG = 13.680570540745645

DRONE_EAST  = 98.00
DRONE_NORTH = -104.06
EST_ALTITUDE = 23.7
EST_YAW = -24.0

# Convert to GPS
est_lat, est_lng = local_m_to_latlng(DRONE_EAST, DRONE_NORTH, REF_LAT, REF_LNG)
print(f"Estimated drone position: lat={est_lat:.7f}, lon={est_lng:.7f}")
print(f"Altitude: {EST_ALTITUDE} m, heading: {EST_YAW}°\n")

all_matches = []

for csv_path in TELEMETRY_FILES:
    fname = csv_path.split("/")[-1]
    telemetry = pd.read_csv(csv_path)[["time_s", "lat", "lng", "altitude_m", "yaw_deg"]].dropna()

    east_t, north_t = latlng_to_local_m(telemetry["lat"].values, telemetry["lng"].values, REF_LAT, REF_LNG)
    pos_dist = np.hypot(east_t - DRONE_EAST, north_t - DRONE_NORTH)
    alt_diff = np.abs(telemetry["altitude_m"].values - EST_ALTITUDE)
    yaw_diff = np.abs(((telemetry["yaw_deg"].values - EST_YAW + 180) % 360) - 180)

    combined = pos_dist * 1.5 + alt_diff * 1.0 + yaw_diff * 0.3

    best_idx = np.argmin(combined)
    best_row = telemetry.iloc[best_idx]

    print(f"{fname}")
    print(f"  t={best_row['time_s']:.1f}s  |  position={pos_dist[best_idx]:.1f}m  |  "
          f"height_diff={alt_diff[best_idx]:.1f}m  |  course_diff={yaw_diff[best_idx]:.1f}°")
    print()

    all_matches.append({
        "file": fname, "time_s": best_row["time_s"],
        "pos_dist": pos_dist[best_idx], "alt_diff": alt_diff[best_idx],
        "yaw_diff": yaw_diff[best_idx], "combined": combined[best_idx],
    })

all_matches.sort(key=lambda x: x["combined"])
for m in all_matches:
    print(f"  {m['file']}: t={m['time_s']:.1f}s, position={m['pos_dist']:.1f}m, "
          f"height_diff={m['alt_diff']:.1f}m, course_diff={m['yaw_diff']:.1f}°")

best = all_matches[0]
print(f"\n Best: {best['file']} (t={best['time_s']:.1f}s)")

In [ ]:
# Multi-frame calibration (bundle adjustment)
import numpy as np
import pandas as pd
from scipy.optimize import least_squares

IMG_WIDTH, IMG_HEIGHT = 1920, 1080
VIDEO_FPS = 60

TELEMETRY_CSV = "/kaggle/input/datasets/natair/djiflightrecord-2026-04-02-17-50-03-csv/DJIFlightRecord_2026-04-02_17-50-03.csv"
VIDEO_OFFSET_S = -6.65   # telemetry_time = video_time + offset

FIELD_CORNERS = {
    "distant right": {"lat": 52.75236395813839, "lon": 13.680075477949087},
    "close left": {"lat": 52.75073848839966, "lon": 13.6822158755006},
    "close right": {"lat": 52.75142022264314, "lon": 13.68265678136193},
    "distant left": {"lat": 52.75163129122488, "lon": 13.6794141191636},
}

OBSERVATIONS = [
    {"frame": 741, "corner": "distant right", "px": 500,  "py": 220},
    {"frame": 741, "corner": "close left", "px": 1050, "py": 630},
    {"frame": 741, "corner": "close right", "px": 1750, "py": 700},
    {"frame": 741, "corner": "distant left", "px": 0,    "py": 250},

    {"frame": 1100, "corner": "distant right", "px": 50, "py": 200},
    {"frame": 1100, "corner": "close right", "px": 1625, "py": 650},

]

telemetry = pd.read_csv(TELEMETRY_CSV)[["time_s", "lat", "lng", "altitude_m", "yaw_deg"]].dropna()

def get_drone_state(video_frame):
    t = video_frame / VIDEO_FPS + VIDEO_OFFSET_S
    idx = (telemetry["time_s"] - t).abs().idxmin()
    row = telemetry.loc[idx]
    return row["lat"], row["lng"], row["altitude_m"], row["yaw_deg"]

def latlng_to_local_m(lat, lng, ref_lat, ref_lng):
    m_per_deg_lat = 111320.0
    m_per_deg_lng = 111320.0 * np.cos(np.radians(ref_lat))
    east = (lng - ref_lng) * m_per_deg_lng
    north = (lat - ref_lat) * m_per_deg_lat
    return east, north

REF_LAT = np.mean([c["lat"] for c in FIELD_CORNERS.values()])
REF_LNG = np.mean([c["lon"] for c in FIELD_CORNERS.values()])

def project_ray_to_ground(drone_east, drone_north, altitude, yaw_deg, pitch_down_deg, focal_px, px, py):
    yaw = np.radians(yaw_deg)
    pitch = np.radians(pitch_down_deg)
    forward = np.array([np.sin(yaw) * np.cos(pitch), np.cos(yaw) * np.cos(pitch), -np.sin(pitch)])
    right   = np.array([np.cos(yaw), -np.sin(yaw), 0])
    down    = np.cross(forward, right)
    dx_px, dy_px = px - IMG_WIDTH / 2, py - IMG_HEIGHT / 2
    direction = right * (dx_px / focal_px) + down * (dy_px / focal_px) + forward
    direction = direction / np.linalg.norm(direction)
    if direction[2] >= -1e-6:
        return None
    t = -altitude / direction[2]
    return drone_east + t * direction[0], drone_north + t * direction[1]

def residuals(params):
    pitch_down_deg, focal_px = params
    errs = []
    for obs in OBSERVATIONS:
        lat, lng, alt, yaw = get_drone_state(obs["frame"])
        drone_east, drone_north = latlng_to_local_m(lat, lng, REF_LAT, REF_LNG)

        proj = project_ray_to_ground(drone_east, drone_north, alt, yaw, pitch_down_deg, focal_px,
                                      obs["px"], obs["py"])
        if proj is None:
            errs.extend([1000, 1000])
            continue
        pred_east, pred_north = proj

        corner = FIELD_CORNERS[obs["corner"]]
        true_east, true_north = latlng_to_local_m(corner["lat"], corner["lon"], REF_LAT, REF_LNG)
        errs.extend([pred_east - true_east, pred_north - true_north])
    return errs

best_result, best_rmse = None, np.inf
for pitch_guess in [10, 20, 30, 45, 60, 75, 90]:
    result = least_squares(residuals, x0=[pitch_guess, 1200.0], max_nfev=20000,
                            bounds=([5, 200], [90, 5000]))
    final_errors = np.array(residuals(result.x)).reshape(-1, 2)
    rmse_m = np.sqrt((final_errors**2).sum(axis=1)).mean()
    print(f"  pitch_guess={pitch_guess}°; pitch={result.x[0]:.1f}°, focal={result.x[1]:.0f}px, RMSE={rmse_m:.2f}m")
    if rmse_m < best_rmse:
        best_rmse, best_result = rmse_m, result

pitch_down_deg, focal_px = best_result.x
print(f"  Camera pitch: {pitch_down_deg:.1f}°")
print(f"  Focus:         {focal_px:.0f}px")
print(f"  RMSE:          {best_rmse:.2f}m  (based on {len(OBSERVATIONS)} observations")

In [ ]:
import numpy as np

def predict_corner_pixel(drone_east, drone_north, altitude, yaw_deg, pitch_down_deg, focal_px,
                          target_east, target_north, img_width=1920, img_height=1080):
    """
    Back projection: Given the actual coordinates of a point on the ground and the camera's pose,
    we calculate which pixel in the frame the point should be visible in.
    Returns None if the point is not visible (for example, if it is behind the camera)
    """

    yaw = np.radians(yaw_deg)
    pitch = np.radians(pitch_down_deg)

    forward = np.array([np.sin(yaw) * np.cos(pitch), np.cos(yaw) * np.cos(pitch), -np.sin(pitch)])
    right   = np.array([np.cos(yaw), -np.sin(yaw), 0])
    down    = np.cross(forward, right)

    # The vector from the camera to the target point (a point on the ground, z=0)
    rel = np.array([target_east - drone_east, target_north - drone_north, 0 - altitude])

    comp_forward = np.dot(rel, forward)
    if comp_forward <= 1e-6:
        return None   # the point behind the camera

    comp_right = np.dot(rel, right)
    comp_down  = np.dot(rel, down)

    px = img_width / 2  + focal_px * comp_right / comp_forward
    py = img_height / 2 + focal_px * comp_down  / comp_forward

    if 0 <= px <= img_width and 0 <= py <= img_height:
        return px, py
    return None   # outside the frame

import pandas as pd
from pathlib import Path
import cv2
import matplotlib.pyplot as plt

VIDEO_FPS = 60
IMG_WIDTH, IMG_HEIGHT = 1920, 1080

TELEMETRY_CSV = "/kaggle/input/datasets/natair/djiflightrecord-2026-04-02-17-50-03-csv/DJIFlightRecord_2026-04-02_17-50-03.csv"
VIDEO_OFFSET_S = -6.65

FRAMES_DIR = "/kaggle/input/datasets/natair/chickens-drone-25k/chickens_drone_25k"

PITCH_DOWN_DEG = 34.2
FOCAL_PX = 834

FIELD_CORNERS = {
    "distant right": {"lat": 52.75236395813839, "lon": 13.680075477949087},
    "close left": {"lat": 52.75073848839966, "lon": 13.6822158755006},
    "close right": {"lat": 52.75142022264314, "lon": 13.68265678136193},
    "distant left":  {"lat": 52.75163129122488, "lon": 13.6794141191636},
}

telemetry = pd.read_csv(TELEMETRY_CSV)[["time_s", "lat", "lng", "altitude_m", "yaw_deg"]].dropna()

def get_drone_state(video_frame):
    t = video_frame / VIDEO_FPS + VIDEO_OFFSET_S
    idx = (telemetry["time_s"] - t).abs().idxmin()
    row = telemetry.loc[idx]
    return row["lat"], row["lng"], row["altitude_m"], row["yaw_deg"]

def latlng_to_local_m(lat, lng, ref_lat, ref_lng):
    m_per_deg_lat = 111320.0
    m_per_deg_lng = 111320.0 * np.cos(np.radians(ref_lat))
    east = (lng - ref_lng) * m_per_deg_lng
    north = (lat - ref_lat) * m_per_deg_lat
    return east, north

REF_LAT = np.mean([c["lat"] for c in FIELD_CORNERS.values()])
REF_LNG = np.mean([c["lon"] for c in FIELD_CORNERS.values()])

import re
EXTS = {'.jpg', '.jpeg', '.png'}
def frame_number(path):
    match = re.search(r'(\d+)', path.stem)
    return int(match.group(1)) if match else 0
frame_files = sorted(
    [p for p in Path(FRAMES_DIR).rglob("*") if p.suffix.lower() in EXTS],
    key=frame_number
)
print(f" Frames found: {len(frame_files)}")

def contact_sheet(frame_indices, cols=5):
    rows = (len(frame_indices) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 3*rows))
    axes = axes.flat if rows > 1 else [axes] if cols == 1 else axes

    for ax, idx in zip(axes, frame_indices):
        img = cv2.cvtColor(cv2.imread(str(frame_files[idx])), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(f"Frame {idx}", fontsize=9)
        ax.axis("off")
    for ax in axes[len(frame_indices):]:
        ax.axis("off")

    plt.tight_layout()
    plt.savefig("/kaggle/working/contact_sheet.jpg", dpi=100, bbox_inches="tight")
    plt.show()

contact_sheet(list(range(0, 5000, 500)))

def show_frame_with_predictions(frame_idx):
    fp = frame_files[frame_idx]
    img = cv2.cvtColor(cv2.imread(str(fp)), cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]

    lat, lng, alt, yaw = get_drone_state(frame_idx)
    drone_east, drone_north = latlng_to_local_m(lat, lng, REF_LAT, REF_LNG)

    fig, ax = plt.subplots(figsize=(18, 10))
    ax.imshow(img)

    for x in range(0, W, 100):
        ax.axvline(x, color='yellow', alpha=0.25, linewidth=0.5)
        ax.text(x, 15, str(x), color='red', fontsize=7)
    for y in range(0, H, 100):
        ax.axhline(y, color='yellow', alpha=0.25, linewidth=0.5)
        ax.text(5, y, str(y), color='red', fontsize=7)

    colors = {"distant right": "cyan", "close left": "lime", "close right": "orange", "distant left": "magenta"}
    found_any = False
    for name, corner in FIELD_CORNERS.items():
        target_east, target_north = latlng_to_local_m(corner["lat"], corner["lon"], REF_LAT, REF_LNG)
        pred = predict_corner_pixel(drone_east, drone_north, alt, yaw, PITCH_DOWN_DEG, FOCAL_PX,
                                     target_east, target_north, W, H)
        if pred:
            px, py = pred
            found_any = True
            ax.plot(px, py, 'o', color=colors[name], markersize=14, markeredgecolor='black', markeredgewidth=1.5)
            ax.annotate(f"  {name}", (px, py), color=colors[name], fontsize=11, fontweight='bold')

    title = f"Frame {frame_idx}  | height={alt:.1f}m, course={yaw:.1f}°"
    if not found_any:
        title += " (no known corners should be visible)"
    ax.set_title(title, fontsize=12)

    plt.tight_layout()
    plt.savefig(f"/kaggle/working/frame_{frame_idx}_prediction.jpg", dpi=100, bbox_inches="tight")
    plt.show()

    print(f"Frame {frame_idx}: drone lat={lat:.6f}, lon={lng:.6f}, height={alt:.1f}m, course={yaw:.1f}°")

show_frame_with_predictions(1500)

try:
    from ipywidgets import interact, IntSlider
    interact(show_frame_with_predictions,
             frame_idx=IntSlider(min=0, max=len(frame_files)-1, step=10, value=1500))
except ImportError:
    print("ipywidgets not found")

In [ ]:
# GEO-REFERENCING: calibration + pixel-to-GPS projection
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
from scipy.optimize import least_squares

VIDEO_FPS = 60
IMG_WIDTH, IMG_HEIGHT = 1920, 1080
TELEMETRY_CSV = "DJIFlightRecord_2026-04-02_16-20-13.csv"
TRACKS_XML = "tracks_dense.xml"

telemetry = pd.read_csv(TELEMETRY_CSV, skiprows=0)
telemetry = telemetry[["time_s", "lat", "lng", "altitude_m", "yaw_deg"]].dropna()

def get_drone_state(time_s):
    idx = (telemetry["time_s"] - time_s).abs().idxmin()
    row = telemetry.loc[idx]
    return row["lat"], row["lng"], row["altitude_m"], row["yaw_deg"]

def latlng_to_local_m(lat, lng, ref_lat, ref_lng):
    """East/North in meters relative to the reference point."""
    m_per_deg_lat = 111320.0
    m_per_deg_lng = 111320.0 * np.cos(np.radians(ref_lat))
    east = (lng - ref_lng) * m_per_deg_lng
    north = (lat - ref_lat) * m_per_deg_lat
    return east, north

def local_m_to_latlng(east, north, ref_lat, ref_lng):
    m_per_deg_lat = 111320.0
    m_per_deg_lng = 111320.0 * np.cos(np.radians(ref_lat))
    lat = ref_lat + north / m_per_deg_lat
    lng = ref_lng + east / m_per_deg_lng
    return lat, lng

REF_LAT, REF_LNG = telemetry.iloc[0]["lat"], telemetry.iloc[0]["lng"]

GROUND_CONTROL_POINTS = [
    {"frame": 0,    "px": 960,  "py": 540,  "lat": 52.75115921079829, "lon": 13.680112910919934},
    {"frame": 5000, "px": 340,  "py": 210,  "lat": 52.7513xxx,        "lon": 13.6802xxx},
    {"frame": 12000,"px": 1500, "py": 700,  "lat": 52.7509xxx,        "lon": 13.6795xxx},
]

def residuals(params, gcps):
    focal_px = params[0]
    errs = []
    for gcp in gcps:
        lat, lng, alt, yaw = get_drone_state(gcp["frame"] / VIDEO_FPS)
        dx_px = gcp["px"] - IMG_WIDTH / 2
        dy_px = gcp["py"] - IMG_HEIGHT / 2

        # turn to an east-north heading along the drone's course
        yaw_rad = np.radians(yaw)
        east_offset  = (dx_px * np.cos(yaw_rad) + dy_px * np.sin(yaw_rad)) * alt / focal_px
        north_offset = (-dx_px * np.sin(yaw_rad) + dy_px * np.cos(yaw_rad)) * alt / focal_px

        drone_east, drone_north = latlng_to_local_m(lat, lng, REF_LAT, REF_LNG)
        pred_east  = drone_east + east_offset
        pred_north = drone_north - north_offset   # sign: y decreases in pixels

        true_east, true_north = latlng_to_local_m(gcp["lat"], gcp["lon"], REF_LAT, REF_LNG)
        errs.extend([pred_east - true_east, pred_north - true_north])
    return errs

result = least_squares(residuals, x0=[1200.0], args=(GROUND_CONTROL_POINTS,))
FOCAL_PX = result.x[0]

final_errors = np.array(residuals([FOCAL_PX], GROUND_CONTROL_POINTS)).reshape(-1, 2)
rmse_m = np.sqrt((final_errors**2).sum(axis=1)).mean()

print(f" Effective focal length: {FOCAL_PX:.1f} px")
print(f" Average error at control points: {rmse_m:.2f} meters")

def pixel_to_latlng(px, py, time_s):
    lat, lng, alt, yaw = get_drone_state(time_s)
    dx_px = px - IMG_WIDTH / 2
    dy_px = py - IMG_HEIGHT / 2
    yaw_rad = np.radians(yaw)
    east_offset  = (dx_px * np.cos(yaw_rad) + dy_px * np.sin(yaw_rad)) * alt / FOCAL_PX
    north_offset = (-dx_px * np.sin(yaw_rad) + dy_px * np.cos(yaw_rad)) * alt / FOCAL_PX

    drone_east, drone_north = latlng_to_local_m(lat, lng, REF_LAT, REF_LNG)
    return local_m_to_latlng(drone_east + east_offset, drone_north - north_offset, REF_LAT, REF_LNG)

tree = ET.parse(TRACKS_XML)
root = tree.getroot()

results = []
for track in root.findall(".//track"):
    track_id = track.get("id")
    for box in track.findall("box"):
        if box.get("outside") == "1":
            continue
        frame = int(box.get("frame"))
        cx = (float(box.get("xtl")) + float(box.get("xbr"))) / 2
        cy = (float(box.get("ytl")) + float(box.get("ybr"))) / 2
        time_s = frame / VIDEO_FPS
        lat, lon = pixel_to_latlng(cx, cy, time_s)
        results.append({"track_id": track_id, "frame": frame, "time_s": time_s, "lat": lat, "lon": lon})

df_geo = pd.DataFrame(results)
df_geo.to_csv("/kaggle/working/chicken_gps_tracks.csv", index=False)
print(f"\n Geolocated detections: {len(df_geo)}")
print(f" Unique tracks: {df_geo["track_id"].nunique()}")